# Application security testing (SAST/DAST)  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 5.5.c: Static-analysis scanner (SAST)

**Skill:** Detect Attacks, Mitigate Risk · **Maps to:** 5.5.B, 5.5.C · **Language:** Python 3 (pytest)

## Problem
Build a miniature SAST rule engine. Given the source lines of a file (as a list of strings, one
per line, 1-indexed), scan each line for known dangerous patterns and return a list of findings, exactly what a real static-analysis tool's rule engine does before a human triages the results.

Detect these categories:
- `"sql-injection-risk"`, a SQL keyword (`SELECT`, `INSERT`, `UPDATE`, `DELETE`) combined with
  string concatenation (`+`) or an f-string/`.format(`/`%` on the same line (i.e., the query is
  being built dynamically instead of parameterized).
- `"eval-use"`, the line calls `eval(` or `exec(`.
- `"hardcoded-secret"`, the line assigns a literal string to a variable whose name contains
  `password`, `secret`, `api_key`, or `token` (case-insensitive), e.g. `password = "hunter2"`.
  An assignment that reads from an environment variable or config object (no literal string on the
  right-hand side) is NOT a finding.

## Function contract
Implement `solve(lines: list) -> list` in the code editor (see the Reference section below).

## Input
- `lines`: `list[str]`, source code lines, in file order. Line numbers are 1-indexed (the first
  element of the list is line 1).

## Output
A `list[dict]`, each `{"line": int, "category": str}`, **sorted by line number ascending**, then by
category name ascending for findings on the same line. If a line matches more than one category,
emit one finding per matching category. If no lines match, return `[]`.

## Examples
```
lines = [
  'query = "SELECT * FROM users WHERE name = \'" + user_input + "\'"',
  'password = "hunter2"',
  'eval(user_expr)',
]
solve(lines) -> [
  {"line": 1, "category": "sql-injection-risk"},
  {"line": 2, "category": "hardcoded-secret"},
  {"line": 3, "category": "eval-use"},
]
```

```
lines = ['password = os.environ["DB_PASSWORD"]', 'db.query(stmt, params)']
solve(lines) -> []
```

## Constraints
- Deterministic, offline, standard library only.
- Findings are sorted by `line`, then `category` (both ascending).
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(lines: list) -> list
```

### Input (`lines`)
`list[str]`, source-code lines, in file order (position 1 = first line, i.e. 1-indexed).

### Output (`result`)
`list[dict]`, each `{"line": int, "category": str}`, sorted by `line` ascending, then by
`category` ascending for findings on the same line. A single line may contribute more than one
finding if it matches more than one category.

### Categories
- `"sql-injection-risk"`, a SQL keyword (`SELECT`/`INSERT`/`UPDATE`/`DELETE`, case-insensitive)
  appears on a line that also builds the string dynamically (`+` concatenation, an f-string,
  `.format(`, or `%` formatting) rather than using a parameterized placeholder.
- `"eval-use"`, the line calls `eval(` or `exec(`.
- `"hardcoded-secret"`, the line assigns a quoted string literal to a variable whose name
  contains `password`, `secret`, `api_key`, or `token` (case-insensitive). An assignment that
  reads from an environment variable or config object (no literal string on the right-hand side)
  is NOT a finding.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Parameterized queries (e.g. `execute("... WHERE id = ?", (x))`) are NOT flagged.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 5.5.c. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
import re
from typing import Dict, List

_SQL_KEYWORD = re.compile(r"\b(SELECT|INSERT|UPDATE|DELETE)\b", re.IGNORECASE)
_DYNAMIC_BUILD = re.compile(r"\+|f[\"']|\.format\(|%\s*[\"']|[\"']\s*%")
_EVAL_USE = re.compile(r"\b(eval|exec)\s*\(")
_SECRET_ASSIGN = re.compile(
    r"\b\w*(password|secret|api_key|token)\w*\s*=\s*[\"']", re.IGNORECASE
)


def solve(lines: List[str]) -> List[Dict[str, object]]:
    """Scan source lines for known dangerous patterns (miniature SAST rule engine).

    Returns findings sorted by line ascending, then category ascending; a line may
    contribute more than one finding.
    """
    findings = []
    for i, line in enumerate(lines, start=1):
        cats = set()
        if _SQL_KEYWORD.search(line) and _DYNAMIC_BUILD.search(line):
            cats.add("sql-injection-risk")
        if _EVAL_USE.search(line):
            cats.add("eval-use")
        if _SECRET_ASSIGN.search(line):
            cats.add("hardcoded-secret")
        for cat in cats:
            findings.append({"line": i, "category": cat})
    findings.sort(key=lambda f: (f["line"], f["category"]))
    return findings


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

import os


# ---- visible cases -------------------------------------------------------
def test_mixed_findings():
    lines = [
        'query = "SELECT * FROM users WHERE name = \'" + user_input + "\'"',
        'password = "hunter2"',
        'eval(user_expr)',
    ]
    assert impl.solve(lines) == [
        {"line": 1, "category": "sql-injection-risk"},
        {"line": 2, "category": "hardcoded-secret"},
        {"line": 3, "category": "eval-use"},
    ]


def test_safe_lines_no_findings():
    lines = ['password = os.environ["DB_PASSWORD"]', 'db.query(stmt, params)']
    assert impl.solve(lines) == []


def test_clean_code_no_findings():
    lines = ["def add(a, b):", "    return a + b", "", "result = add(1, 2)"]
    assert impl.solve(lines) == []


def test_parameterized_sql_not_flagged():
    lines = ['cur.execute("SELECT * FROM users WHERE id = ?", (user_id,))']
    assert impl.solve(lines) == []


def test_exec_flagged_same_as_eval():
    lines = ["exec(payload)"]
    assert impl.solve(lines) == [{"line": 1, "category": "eval-use"}]


# ---- hidden / edge cases -------------------------------------------------
def test_multiple_categories_same_line():
    lines = ['token = "abc123"; eval(token)']
    result = impl.solve(lines)
    assert result == [
        {"line": 1, "category": "eval-use"},
        {"line": 1, "category": "hardcoded-secret"},
    ]


def test_case_insensitive_secret_keyword_and_sql():
    lines = ['API_KEY = "sk_live_xyz"', 'q = "select * from t where x=" + v']
    assert impl.solve(lines) == [
        {"line": 1, "category": "hardcoded-secret"},
        {"line": 2, "category": "sql-injection-risk"},
    ]


def test_fstring_sql_flagged():
    lines = ['q = f"SELECT * FROM t WHERE id={uid}"']
    assert impl.solve(lines) == [{"line": 1, "category": "sql-injection-risk"}]


def test_empty_input():
    assert impl.solve([]) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
